In [ ]:
%%capture
import os
import pandas as pd
from dj_notebook import activate
from pathlib import Path

env_file = os.environ["INTECOMM_ENV"]
analysis_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
reports_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
plus = activate(dotenv_file=env_file)


In [ ]:
from PIL import Image
from great_tables import GT, html, style, loc
from intecomm_analytics.dataframes import get_df_main_1858
from intecomm_analytics.constants import UNDEFINED, DM_ALONE, HTN_ALONE, HIV_ALONE, HTN_DM
from intecomm_analytics.notebooks.primary.table_utils import get_primary_cohorts_by_categorical_column
from intecomm_rando.constants import COMMUNITY_ARM, FACILITY_ARM


In [ ]:
df_main = get_df_main_1858(None)


In [ ]:
# primary_cohort
tbl_dct = get_primary_cohorts_by_categorical_column(df_main, "primary_cohort")
dftbl = pd.DataFrame(tbl_dct)
mapping = {DM_ALONE:"Diabetes alone", HTN_ALONE:"Hypertension alone", HTN_DM:"Diabetes and hypertension", HIV_ALONE:"HIV alone", UNDEFINED:"UNDEFINED", "n":"n"}
dftbl["Statistics"] = dftbl["Statistics"].map(mapping)
dftbl = dftbl[dftbl["Statistics"]!="UNDEFINED"]
dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=["n", "Diabetes alone", "Hypertension alone", "Diabetes and hypertension", "HIV alone"], ordered=True)
dftbl.sort_values(by=["Statistics"], ascending=True, inplace=True)
dftbl.replace("0 (0.0%)", "NA", inplace=True)
dfcond = dftbl.copy()


In [ ]:
# country
tbl_dct = get_primary_cohorts_by_categorical_column(df_main, "country")
dftbl = pd.DataFrame(tbl_dct)
mapping = {"n": "n", "TZ": "Tanzania", "UG": "Uganda"}
dftbl["Statistics"] = dftbl["Statistics"].map(mapping)
dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=["n", "Tanzania", "Uganda"], ordered=True)
dftbl = dftbl.sort_values(by=["Statistics"], ascending=True)
dfcountry = dftbl.copy()


In [ ]:
# gender
tbl_dct = get_primary_cohorts_by_categorical_column(df_main, "gender")
dftbl = pd.DataFrame(tbl_dct)
dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=["n", "Female", "Male"], ordered=True)
dftbl = dftbl.sort_values(by=["Statistics"], ascending=True)
dfgender = dftbl.copy()


In [ ]:
# education
df1 = df_main.copy()
df1["education"] = df1["education"].apply(lambda x: "missing" if pd.isna(x) else x)
mapping = {
    "n": "n",
    "no_formal_education": "no_formal_education",
    "primary": "primary",
    "secondary": "secondary_or_tertiary",
    "post_secondary": "secondary_or_tertiary",
    "tertiary": "secondary_or_tertiary",
    "missing": "missing"}
df1["education"] = df1["education"].map(mapping)
tbl_dct = get_primary_cohorts_by_categorical_column(df1, "education")
dftbl = pd.DataFrame(tbl_dct)
mapping = {
    "n": "n",
    "no_formal_education": "No formal education",
    "primary": "Primary",
    "secondary_or_tertiary": "Secondary or tertiary",
    "missing": "Missing"}
dftbl["Statistics"] = dftbl["Statistics"].map(mapping)
dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=["n", "No formal education", "Primary", "Secondary or tertiary", "Missing"], ordered=True)
dftbl = dftbl.sort_values(by=["Statistics"], ascending=True)
dftbl = dftbl.reset_index(drop=True)
dfed = dftbl.copy()

In [ ]:
# age_in_years
df1 = df_main.copy()
bins = [0,34, 49, 110]
labels = ["<35", "35-49", ">=50"]
df1["age"] = pd.cut(df1["age_in_years"], bins, labels=labels)
tbl_dct = get_primary_cohorts_by_categorical_column(df1, "age")
dftbl = pd.DataFrame(tbl_dct)
data = ["Mean, SD"]
for col in ["ncd", "hiv_only"]:
    for arm in [COMMUNITY_ARM, FACILITY_ARM]:
        data.append(
            f"{round(df1[(df1.assignment==arm) & (getattr(df1, col)==1)]['age_in_years'].mean(),1)} "
            f"({round(df1[(df1.assignment==arm) & (getattr(df1, col)==1)]['age_in_years'].std(),1)})"
        )
dftbl.loc[len(dftbl)] = data
dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=["n", "Mean, SD", "<35", "35-49", ">=50"], ordered=True)
dftbl = dftbl.sort_values(by=["Statistics"], ascending=True)
dftbl = dftbl.reset_index(drop=True)
dfage= dftbl.copy()


In [ ]:
# bp_controlled_endline
df1 = df_main.copy()
cond = ((df1.htn==1) & (df1.hiv==0))
label = "<140/90 mm Hg among<BR>&nbsp;&nbsp;&nbsp;&nbsp;participants with<BR>&nbsp;&nbsp;&nbsp;&nbsp;hypertension"
df1.loc[cond, "bp_controlled_endline"] = df1.loc[cond, "bp_controlled_endline"].fillna(-1)
tbl_dct = get_primary_cohorts_by_categorical_column(df1[cond], "bp_controlled_endline")
dftbl = pd.DataFrame(tbl_dct)
mapping = {"n": "n", -1: "Missing", 1: label, 0: "Uncontrolled"}
dftbl["Statistics"] = dftbl["Statistics"].map(mapping)
dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=["n", label, "Uncontrolled", "Missing"], ordered=True)
dftbl = dftbl.sort_values(by=["Statistics"], ascending=True)
dftbl = dftbl.reset_index(drop=True)
for col in ["Community Ncd", "Facility Ncd"]:
    value = dftbl.loc[1, col].split(" ")
    value = [value[0], "/", str(dftbl.loc[0, col]), " ", value[1]]
    value = "".join(value)
    dftbl.loc[1, col] = value
dftbl.replace("0 (0.0%)", "NA", inplace=True)
dftbl.drop(0, inplace=True)
dfbp = dftbl.copy()


In [ ]:
# glucose_controlled_endline
df1 = df_main.copy()
cond = ((df1.dm==1) & (df1.hiv==0))
label = "<7.0 mmol/L among<BR>&nbsp;&nbsp;&nbsp;&nbsp;participants with<BR>&nbsp;&nbsp;&nbsp;&nbsp;diabetes"
df1.loc[cond, "glucose_controlled_endline"] = df1.loc[cond, "glucose_controlled_endline"].fillna(-1)
tbl_dct = get_primary_cohorts_by_categorical_column(df1[cond], "glucose_controlled_endline")
dftbl = pd.DataFrame(tbl_dct)
mapping = {"n": "n", -1: "Missing", 1: label, 0: "Uncontrolled"}
dftbl["Statistics"] = dftbl["Statistics"].map(mapping)
dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=["n", label, "Uncontrolled", "Missing"], ordered=True)
dftbl = dftbl.sort_values(by=["Statistics"], ascending=True)
dftbl = dftbl.reset_index(drop=True)
for col in ["Community Ncd", "Facility Ncd"]:
    value = dftbl.loc[1, col].split(" ")
    value = [value[0], "/", str(dftbl.loc[0, col]), " ", value[1]]
    value = "".join(value)
    dftbl.loc[1, col] = value
dftbl.replace("0 (0.0%)", "NA", inplace=True)
dftbl.drop(0, inplace=True)
dfglu = dftbl.copy()

In [ ]:
# vl_endline_suppressed
df1 = df_main.copy()
cond = ((df1.hiv==1) & (df1.dm==0) & (df1.htn==0))
label = "<1000 copies per mL"
df1.loc[cond, "vl_endline_suppressed"] = df1.loc[cond, "vl_endline_suppressed"].fillna(-1)
tbl_dct = get_primary_cohorts_by_categorical_column(df1[cond], "vl_endline_suppressed")
dftbl = pd.DataFrame(tbl_dct)
mapping = {"n": "n", -1: "Missing", 1: label, 0: "Uncontrolled"}
dftbl["Statistics"] = dftbl["Statistics"].map(mapping)
dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=["n", label, "Uncontrolled", "Missing"], ordered=True)
dftbl = dftbl.sort_values(by=["Statistics"], ascending=True)
dftbl = dftbl.reset_index(drop=True)
for col in ["Community Hiv only", "Facility Hiv only"]:
    value = dftbl.loc[1, col].split(" ")
    value = [value[0], "/", str(dftbl.loc[0, col]), " ", value[1]]
    value = "".join(value)
    dftbl.loc[1, col] = value
dftbl.replace("0 (0.0%)", "NA", inplace=True)
dftbl.drop(0, inplace=True)
dfvl = dftbl.copy()

In [ ]:
# concat all
dftbl = pd.concat([dfcountry, dfcond, dfgender, dfed, dfage, dfbp, dfglu, dfvl])
dftbl = dftbl.drop_duplicates(keep='first')
dftbl = dftbl.reset_index(drop=True)

In [ ]:
# convert to GT
tbl_groups = dftbl.assign(group=[""] * 1 + ["Site"] * 2 + ["Condition"] * 4 + ["Sex"] * 2 + ["Education"] * 4 + ["Age"] * 4 + ["Blood pressure"] * 3 + ["Fasting blood glucose"] * 3 + ["HIV viral load"] * 2)
tbl_groups = tbl_groups[tbl_groups['group'] != ""]
tbl_groups['Statistics'] = tbl_groups['Statistics'].apply(lambda x: f'&nbsp;&nbsp;&nbsp;{x}')
table = (GT(tbl_groups)
    .tab_header(title="Table 1: Baseline characteristics")
    .tab_spanner(label=html(f"Participants with diabetes,<BR>hypertension, or both<br> (n={dftbl.loc[0, ["Community Ncd", "Facility Ncd"]].sum()})"), columns=[1,2])
    .tab_spanner(label=html(f'Participants with<BR>HIV alone<BR>(n={dftbl.loc[0, ["Community Hiv only", "Facility Hiv only"]].sum()})'), columns=[3,4])
    .cols_label({
        "Community Ncd": html(f"Community<BR>(n={dftbl.loc[0, ["Community Ncd"]].sum()})"),
        "Facility Ncd": html(f"Facility<br>(n={dftbl.loc[0, ["Facility Ncd"]].sum()})"),
        "Community Hiv only": html(f"Community<br>(n={dftbl.loc[0, ["Community Hiv only"]].sum()})"),
        "Facility Hiv only": html(f"Facility<br>(n={dftbl.loc[0, ["Facility Hiv only"]].sum()})")})
    .cols_align(align='left', columns=[0])
    .cols_align(align='center', columns=[1,2,3,4])
    .tab_stub(rowname_col="Statistics", groupname_col="group")
    .opt_stylize(style=3)
    .opt_row_striping(row_striping=False)
    .opt_vertical_padding(scale=1.2)
    .opt_horizontal_padding(scale=1.0)
    .tab_options(
        stub_background_color='white',
        row_group_border_bottom_style='hidden',
        row_group_padding=0.5,
        row_group_background_color="white",
        table_background_color="white",
        table_font_size=12)
    .tab_style(
        style=[style.fill(color="white"), style.text(color="black")],
        locations=loc.body(columns=[1,2, 3, 4], rows=list(range(0, 25))))
    )
table.show()

In [ ]:
# save as png
table.save(analysis_folder / "baseline_characteristics.png")

In [ ]:
# export to PDF
image = Image.open(analysis_folder / "baseline_characteristics.png")
image = image.resize((image.width * 6, image.height * 6), Image.LANCZOS)
image.save(analysis_folder / "baseline_characteristics.pdf", "PDF", resolution=800, optimize=True, quality=95)